# Pelatihan Paket B: Klasifikasi Retinopati Diabetik (APTOS 2019)

Notebook terpadu untuk menjalankan **Fase P2 (Audit & Deduplikasi)**, **Cache Prapemrosesan `384 x 384`**, **Eksperimen E1 sampai E3**, serta **E4 (Kalibrasi, Evaluasi Held-Out Test, Grad-CAM++, dan Ekspor ONNX)** pada Google Colab atau Kaggle Notebook.

> Gunakan saklar `SMOKE_TEST = True` lebih dulu untuk memverifikasi seluruh alur selama sekitar 1 hingga 2 menit pada subset kecil sebelum menjalankan pelatihan penuh (`SMOKE_TEST = False`, `15` epoch).

In [ ]:
import os
import sys
from pathlib import Path

SMOKE_TEST = True
FORCE_RETRAIN = False
INPUT_SIZE = 384
EPOCHS = 2 if SMOKE_TEST else 15
BATCH_SIZE = 16
VAL_FOLD = 1
SEED = 42

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_COLAB and not IS_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Visikom-UAS-Mydria/outputs")
    WORK_ROOT = Path("/content/mydria_work")
elif IS_KAGGLE:
    OUTPUT_ROOT = Path("/kaggle/working/outputs")
    WORK_ROOT = Path("/tmp/visikom_work")
else:
    OUTPUT_ROOT = Path("outputs")
    WORK_ROOT = Path("work_cache")

RUN_ROOT = OUTPUT_ROOT / ("smoke" if SMOKE_TEST else "full")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = WORK_ROOT / "cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"IS_COLAB={IS_COLAB}, IS_KAGGLE={IS_KAGGLE}, SMOKE_TEST={SMOKE_TEST}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT} | RUN_ROOT={RUN_ROOT}")

!pip install -q kagglehub timm albumentations onnx onnxruntime scipy

In [ ]:
import json as _json
import subprocess
import zipfile

def _setup_kaggle_auth() -> None:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    drive_kaggle = Path("/content/drive/MyDrive/Visikom-UAS-Mydria/kaggle.json")
    if not kaggle_json.exists() and IS_COLAB:
        if drive_kaggle.exists():
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(drive_kaggle.read_bytes())
            os.chmod(kaggle_json, 0o600)
        else:
            try:
                from google.colab import userdata
                k_user = userdata.get("KAGGLE_USERNAME")
                k_key = userdata.get("KAGGLE_KEY")
                k_tok = None
                try:
                    k_tok = userdata.get("KAGGLE_API_TOKEN")
                except Exception:
                    pass
                if k_tok:
                    os.environ["KAGGLE_API_TOKEN"] = k_tok
                if k_user and k_key:
                    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
                    kaggle_json.write_text(_json.dumps({"username": k_user, "key": k_key}), encoding="utf-8")
                    os.chmod(kaggle_json, 0o600)
            except Exception:
                pass
        if not kaggle_json.exists() and "KAGGLE_API_TOKEN" not in os.environ:
            from google.colab import files
            print("Unggah berkas kaggle.json dari akun Kaggle Anda (pastikan sudah klik 'I Understand and Accept' di aturan kompetisi APTOS 2019).")
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Tidak ada berkas kaggle.json yang diunggah.")
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(next(iter(uploaded.values())))
            os.chmod(kaggle_json, 0o600)

    if kaggle_json.exists():
        creds = _json.loads(kaggle_json.read_text(encoding="utf-8"))
        if "username" in creds and "key" in creds:
            os.environ["KAGGLE_USERNAME"] = str(creds["username"])
            os.environ["KAGGLE_KEY"] = str(creds["key"])
            print(f"Autentikasi Kaggle aktif untuk akun: {creds['username']}")
        elif "token" in creds:
            os.environ["KAGGLE_API_TOKEN"] = str(creds["token"])
            print("Autentikasi KAGGLE_API_TOKEN aktif.")
        else:
            raise ValueError("Format kaggle.json tidak valid: wajib memuat 'username' dan 'key'.")

KAGGLE_INPUT = Path("/kaggle/input/aptos2019-blindness-detection")
if IS_KAGGLE and not (KAGGLE_INPUT / "train.csv").exists():
    found_csv = next(Path("/kaggle/input").rglob("train.csv"), None)
    if found_csv is not None:
        KAGGLE_INPUT = found_csv.parent
if IS_KAGGLE and (KAGGLE_INPUT / "train.csv").exists():
    DATA_DIR = KAGGLE_INPUT
else:
    DATA_DIR = WORK_ROOT / "aptos2019"
    stale_zip = DATA_DIR / "aptos2019-blindness-detection.zip"
    if stale_zip.exists() and not zipfile.is_zipfile(stale_zip):
        preview = stale_zip.read_bytes()[:250].decode("utf-8", errors="replace")
        print(f"Menghapus zip rusak sisa percobaan sebelumnya ({stale_zip.stat().st_size} byte): {preview!r}")
        stale_zip.unlink(missing_ok=True)

    if not (DATA_DIR / "train.csv").exists() or not (DATA_DIR / "train_images").exists():
        _setup_kaggle_auth()
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        try:
            import kagglehub
            kh_dir = Path(kagglehub.competition_download("aptos2019-blindness-detection"))
            print("Unduhan kagglehub selesai di:", kh_dir)
            if (kh_dir / "train.csv").exists():
                for sub_name in ("train_images.zip", "train.zip"):
                    nz = kh_dir / sub_name
                    if nz.exists() and not (kh_dir / "train_images").exists():
                        (kh_dir / "train_images").mkdir(parents=True, exist_ok=True)
                        with zipfile.ZipFile(nz, "r") as zf:
                            zf.extractall(kh_dir / "train_images")
                if (kh_dir / "train_images").exists():
                    DATA_DIR = kh_dir
        except Exception as kh_err:
            err_str = str(kh_err)
            if "403" in err_str or "401" in err_str or "Forbidden" in err_str:
                raise RuntimeError(
                    "Akses ditolak Kaggle (401/403). Pastikan Anda sudah membuka "
                    "https://www.kaggle.com/competitions/aptos2019-blindness-detection/rules "
                    "lalu klik 'I Understand and Accept' menggunakan akun Kaggle pada kaggle.json Anda."
                ) from kh_err
            print(f"Fallback ke kaggle CLI karena: {kh_err}")

        if not (DATA_DIR / "train.csv").exists() or not (DATA_DIR / "train_images").exists():
            zip_path = DATA_DIR / "aptos2019-blindness-detection.zip"
            subprocess.run(
                ["kaggle", "competitions", "download", "-c", "aptos2019-blindness-detection", "-p", str(DATA_DIR), "--force"],
                check=True,
            )
            size_bytes = zip_path.stat().st_size if zip_path.exists() else 0
            print(f"Ukuran arsip terunduh: {size_bytes / (1024 * 1024):.2f} MB")
            if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
                preview = zip_path.read_bytes()[:300].decode("utf-8", errors="replace") if zip_path.exists() else "<kosong>"
                zip_path.unlink(missing_ok=True)
                raise RuntimeError(
                    f"Berkas unduhan bukan arsip ZIP valid (ukuran={size_bytes} byte, cuplikan={preview!r}). "
                    "Pastikan aturan kompetisi APTOS 2019 sudah disetujui di Kaggle dan token kaggle.json aktif."
                )
            with zipfile.ZipFile(zip_path, "r") as zf:
                for member in zf.namelist():
                    if member.startswith("test_images"):
                        continue
                    zf.extract(member, DATA_DIR)
            zip_path.unlink(missing_ok=True)
            for sub_name in ("train_images.zip", "train.zip"):
                nz = DATA_DIR / sub_name
                if nz.exists() and not (DATA_DIR / "train_images").exists():
                    (DATA_DIR / "train_images").mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(nz, "r") as zf:
                        zf.extractall(DATA_DIR / "train_images")
                    nz.unlink(missing_ok=True)

print("DATA_DIR:", DATA_DIR, "| train.csv:", (DATA_DIR / "train.csv").exists(), "| train_images:", (DATA_DIR / "train_images").exists())

In [ ]:
REPO_URL = "https://github.com/feb027/Visikom-UAS.git"
DRIVE_REPO = Path("/content/drive/MyDrive/Visikom-UAS-Mydria/Visikom-UAS")

if Path("src/visikom_dr").exists():
    REPO_DIR = Path.cwd()
elif (WORK_ROOT / "Visikom-UAS/src/visikom_dr").exists():
    REPO_DIR = WORK_ROOT / "Visikom-UAS"
    !git -C {REPO_DIR} pull --ff-only
elif REPO_URL:
    REPO_DIR = WORK_ROOT / "Visikom-UAS"
    !git clone {REPO_URL} {REPO_DIR}
elif DRIVE_REPO.exists():
    REPO_DIR = DRIVE_REPO
else:
    raise FileNotFoundError(
        "Folder src/visikom_dr tidak ditemukan. Isi REPO_URL atau salin repositori ke Google Drive."
    )

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import json
import pandas as pd
from visikom_dr.audit_dataset import audit_aptos_directory

SPLITS_CSV = OUTPUT_ROOT / "aptos_splits.csv"
AUDIT_JSON = OUTPUT_ROOT / "aptos_audit_summary.json"

if SPLITS_CSV.exists():
    split_df = pd.read_csv(SPLITS_CSV)
    audit_summary = json.loads(AUDIT_JSON.read_text(encoding="utf-8")) if AUDIT_JSON.exists() else {}
else:
    split_df, audit_summary = audit_aptos_directory(
        data_dir=DATA_DIR,
        output_csv=SPLITS_CSV,
        summary_json=AUDIT_JSON,
        n_folds=5,
        holdout_fold=0,
        seed=SEED,
    )

print(json.dumps(audit_summary, indent=2))
if SMOKE_TEST:
    active_df = split_df.groupby(["fold", "diagnosis"], group_keys=False).head(3).reset_index(drop=True)
    print(f"Mode SMOKE_TEST aktif: menggunakan {len(active_df)} citra contoh lintas fold {sorted(active_df['fold'].unique().tolist())}.")
else:
    active_df = split_df

In [ ]:
from visikom_dr.train import prepare_preprocessed_cache

for mode in ("raw_resize", "crop_only", "lab_clahe", "ben_graham"):
    cached = prepare_preprocessed_cache(
        split_df=active_df,
        raw_images_dir=DATA_DIR / "train_images",
        cache_dir=CACHE_ROOT,
        mode=mode,
        size=INPUT_SIZE,
    )
    print(f"Cache siap: {cached}")

In [ ]:
from visikom_dr.train import EXPERIMENT_PRESETS, TrainConfig, run_training_experiment

def run_or_resume(cfg: TrainConfig):
    out_dir = RUN_ROOT / cfg.experiment_id
    summary_path = out_dir / "metrics_summary.json"
    if not SMOKE_TEST and not FORCE_RETRAIN and summary_path.exists():
        print(f"Memuat hasil tersimpan untuk {cfg.experiment_id} dari {summary_path}")
        return json.loads(summary_path.read_text(encoding="utf-8"))
    return run_training_experiment(
        config=cfg,
        split_df=active_df,
        raw_images_dir=DATA_DIR / "train_images",
        output_dir=out_dir,
        cache_dir=CACHE_ROOT,
    )

e1_rows = []
for preset_id in ("E1a_raw", "E1b_crop", "E1c_clahe", "E1d_bengraham"):
    cfg = TrainConfig(
        experiment_id=preset_id,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        val_fold=VAL_FOLD,
        input_size=INPUT_SIZE,
        seed=SEED,
        **EXPERIMENT_PRESETS[preset_id],
    )
    res = run_or_resume(cfg)
    vm = res["val_metrics"]
    e1_rows.append(
        {
            "experiment": preset_id,
            "preprocess_mode": cfg.preprocess_mode,
            "val_qwk": vm["qwk"],
            "val_macro_f1": vm["macro_f1"],
            "mild_recall": vm["per_class"]["1"]["recall"],
            "val_accuracy": vm["accuracy"],
        }
    )

e1_df = pd.DataFrame(e1_rows).sort_values("val_qwk", ascending=False)
e1_df.to_csv(RUN_ROOT / "E1_comparison.csv", index=False)
BEST_PREPROCESS = str(e1_df.iloc[0]["preprocess_mode"])
print(e1_df.to_string(index=False))
print(f"Mode prapemrosesan terpilih dari E1: {BEST_PREPROCESS}")

In [ ]:
e2_rows = []
for preset_id in ("E2_resnet50", "E2_effb0", "E2_convnextv2"):
    params = dict(EXPERIMENT_PRESETS[preset_id])
    params["preprocess_mode"] = BEST_PREPROCESS
    cfg = TrainConfig(
        experiment_id=preset_id,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        val_fold=VAL_FOLD,
        input_size=INPUT_SIZE,
        seed=SEED,
        **params,
    )
    res = run_or_resume(cfg)
    vm = res["val_metrics"]
    e2_rows.append(
        {
            "experiment": preset_id,
            "backbone": cfg.backbone,
            "val_qwk": vm["qwk"],
            "val_macro_f1": vm["macro_f1"],
            "referable_auroc": vm["referable_dr"]["auroc"],
            "val_accuracy": vm["accuracy"],
        }
    )

e2_df = pd.DataFrame(e2_rows).sort_values("val_qwk", ascending=False)
e2_df.to_csv(RUN_ROOT / "E2_comparison.csv", index=False)
BEST_BACKBONE = str(e2_df.iloc[0]["backbone"])
print(e2_df.to_string(index=False))
print(f"Backbone terpilih dari E2: {BEST_BACKBONE}")

In [ ]:
BACKBONE_DEFAULT_LR = {"resnet50": 2e-4, "efficientnet_b0": 3e-4, "convnextv2_tiny": 1e-4}
e3_rows = []
e3_configs = {}
for preset_id in ("E3a_ce_plain", "E3b_ce_single_weight", "E3c_corn", "E3d_smooth_l1"):
    params = dict(EXPERIMENT_PRESETS[preset_id])
    params["backbone"] = BEST_BACKBONE
    params["preprocess_mode"] = BEST_PREPROCESS
    params["lr"] = BACKBONE_DEFAULT_LR.get(BEST_BACKBONE, params.get("lr", 2e-4))
    cfg = TrainConfig(
        experiment_id=preset_id,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        val_fold=VAL_FOLD,
        input_size=INPUT_SIZE,
        seed=SEED,
        **params,
    )
    res = run_or_resume(cfg)
    e3_configs[preset_id] = (cfg, res)
    vm = res["val_metrics"]
    e3_rows.append(
        {
            "experiment": preset_id,
            "head_type": cfg.head_type,
            "val_qwk": vm["qwk"],
            "val_macro_f1": vm["macro_f1"],
            "moderate_f1": vm["per_class"]["2"]["f1"],
            "ece": vm["ece"],
        }
    )

e3_df = pd.DataFrame(e3_rows).sort_values("val_qwk", ascending=False)
e3_df.to_csv(RUN_ROOT / "E3_comparison.csv", index=False)
BEST_E3_ID = str(e3_df.iloc[0]["experiment"])
print(e3_df.to_string(index=False))
print(f"Konfigurasi pemenang E3: {BEST_E3_ID}")

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import torch
from torch.utils.data import DataLoader

from visikom_dr.calibration_xai import (
    compute_gradcam_plusplus,
    export_model_to_onnx,
    overlay_heatmap_on_fundus,
    raw_output_to_predictions,
)
from visikom_dr.metrics import compute_classification_metrics
from visikom_dr.models import RetinaGradingModel
from visikom_dr.preprocessing import preprocess_fundus
from visikom_dr.train import AptosFundusDataset, image_to_tensor

best_cfg, best_res = e3_configs[BEST_E3_ID]
best_dir = RUN_ROOT / BEST_E3_ID
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RetinaGradingModel(
    backbone=best_cfg.backbone,
    head_type=best_cfg.head_type,
    pretrained=False,
    dropout=best_cfg.dropout,
).to(device)
model.load_state_dict(torch.load(best_dir / "best_model.pt", map_location=device, weights_only=True))
model.eval()

test_df = active_df[active_df["split"] == "test"].reset_index(drop=True)
test_ds = AptosFundusDataset(
    test_df,
    raw_images_dir=DATA_DIR / "train_images",
    cache_subdir=CACHE_ROOT / f"{best_cfg.preprocess_mode}_{INPUT_SIZE}",
    preprocess_mode=best_cfg.preprocess_mode,
    input_size=INPUT_SIZE,
    is_train=False,
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

raw_chunks, target_chunks = [], []
with torch.no_grad():
    for imgs, ys in test_loader:
        raw_chunks.append(model(imgs.to(device)).cpu().numpy())
        target_chunks.append(ys.numpy())

test_raw = np.concatenate(raw_chunks, axis=0)
test_targets = np.concatenate(target_chunks, axis=0)
cal_temp = float(best_res["calibration"]["temperature"])
cal_sigma = float(best_res["calibration"]["smooth_sigma"])
cal_thresh = best_res["fitted_thresholds"]

test_probs, test_preds = raw_output_to_predictions(
    test_raw,
    head_type=best_cfg.head_type,
    temperature=cal_temp,
    thresholds=cal_thresh,
    smooth_sigma=cal_sigma,
)
e4_test_metrics = compute_classification_metrics(test_targets, test_preds, probs=test_probs)
(RUN_ROOT / "E4_heldout_test_metrics.json").write_text(
    json.dumps(e4_test_metrics, indent=2), encoding="utf-8"
)
print("Held-Out Test Metrics (Fold 0):")
print(json.dumps({k: e4_test_metrics[k] for k in ("qwk", "accuracy", "macro_f1", "ece", "referable_dr")}, indent=2))

sample_dir = RUN_ROOT / "gradcam_samples"
sample_dir.mkdir(parents=True, exist_ok=True)
for grade in range(5):
    sub = test_df[test_df["diagnosis"] == grade]
    if len(sub) == 0:
        continue
    code = str(sub.iloc[0]["id_code"])
    src_path = DATA_DIR / "train_images" / f"{code}.png"
    if not src_path.exists():
        continue
    rgb = cv2.cvtColor(cv2.imread(str(src_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    proc = preprocess_fundus(rgb, mode=best_cfg.preprocess_mode, size=INPUT_SIZE)
    tensor_in = image_to_tensor(proc).unsqueeze(0).to(device)
    cam = compute_gradcam_plusplus(model, tensor_in)
    overlay = overlay_heatmap_on_fundus(proc, cam)
    cv2.imwrite(str(sample_dir / f"grade_{grade}_{code}_gradcam.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

ONNX_PATH = RUN_ROOT / "model_best.onnx"
META_PATH = RUN_ROOT / "model_meta.json"
model_cpu = model.cpu().eval()
export_model_to_onnx(
    model=model_cpu,
    output_onnx_path=ONNX_PATH,
    meta_json_path=META_PATH,
    preprocess_mode=best_cfg.preprocess_mode,
    input_size=INPUT_SIZE,
    temperature=cal_temp,
    smooth_sigma=cal_sigma,
    thresholds=cal_thresh,
    extra_meta={"experiment_id": BEST_E3_ID, "test_qwk": e4_test_metrics["qwk"], "smoke_test": SMOKE_TEST},
)

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
dummy_np = np.random.randn(1, 3, INPUT_SIZE, INPUT_SIZE).astype(np.float32)
raw_ort, cam_ort = sess.run(None, {"input": dummy_np})
print(f"ONNX terverifikasi: {ONNX_PATH.name} | raw={raw_ort.shape} | cam={cam_ort.shape}")